# B2: FAO Carbon Accounting

Convert FAO production statistics from fresh weight to carbon content (g C m⁻²).

- Input: FAO statistics CSV (`fao/`)
- Conversion: fresh weight → dry matter → carbon
- Output: crop carbon lookup table

In [ ]:
import yaml
from pathlib import Path
import sys
sys.path.append(str(Path('../src')))

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr

with open(Path('../config.yaml')) as f:
    cfg = yaml.safe_load(f)

In [ ]:
for k, v in cfg['paths'].items():
    cfg["paths"][k] = Path(v)#.expanduser().resolve()

cfg["data_root"] = Path(cfg["data_root"])

# FAO accounting

In [ ]:
gdf_world_merged = (
    gpd.read_parquet(cfg["paths"]["output"] / "world_clean.parquet")
        .query("NE_name != 'Antarctica'")
        .reset_index(drop=True)
        .assign(idx=lambda x: np.arange(1, x.shape[0] + 1))    
)

In [ ]:
_years = list(range(2010, 2025))
FAO_m49_lst = gdf_world_merged['FAO_m49'].dropna().unique().tolist()
year_cols = [f"Y{y}" for y in _years]
meta_cols = ["Area Code", "Area Code (M49)", "Area", "Item Code", "code", "Item", "Element Code", "Element", "Unit"]
dic_year_rename = {f"Y{y}": y for y in _years}
dic_FAO2region = gdf_world_merged.set_index("FAO_name")["PNAS_region"].to_dict()

## Crop

In [ ]:
df_crop_classification = pd.read_csv("../data/params/crop_classification.csv", index_col=0)


In [ ]:
df_harv_f = pd.read_excel(Path("../data/params/crop_coefficients.xlsx"), sheet_name="harvest factor")\
    .rename(columns={"harvest factor": "Harvest_cls"})\
    .melt(id_vars="Harvest_cls").rename(columns={"variable": "PNAS_region", "value": "hf"})
    
df_rec_r = pd.read_excel(Path("../data/params/crop_coefficients.xlsx"), sheet_name="recovery rate")\
        .rename(columns={"recovery rate": "Recover_cls"})\
        .melt(id_vars="Recover_cls").rename(columns={"variable": "PNAS_region", "value": "rr"})
        
dic_moisture = pd.read_csv(Path("../data/params/crop_moisture.csv")).set_index("Item")["moisture"].to_dict()

In [ ]:
# keep only the first
# '01921.01', # Seed cotton, unginned
# '01921.02', # Cotton lint, ginned
# '0143', # Cotton seed

# keep only the first
# '01491.01', # Oil palm fruit
# '01491.02', # Palm kernels

exclude_item_codes = ['01921.02', '0143', '01491.02']

In [ ]:
df_crop_all = (
    pd.read_csv(cfg["paths"]["fao"] / "2024/production/Production_Crops_Livestock_E_All_Data_NOFLAG.csv", encoding_errors="ignore")
    .query("Element == 'Production'")
    .query("Unit != '1000 No'")
    .query("`Area Code (M49)` in @FAO_m49_lst")
    .rename(columns={"Item Code (CPC)": "code"})
    .assign(code=lambda x: x["code"].str.replace("'", ""))
    [meta_cols + year_cols]
    .dropna(subset=year_cols, how="all")
    .rename(columns=dic_year_rename)
    .query("code.str.startswith('01')") # 01 Products of agriculture, horticulture and market gardening
    # 011 Cereals
    .query("~code.str.startswith('012')") # 012 Vegetables
    .query("~code.str.startswith('013')") # 013 Fruits and nuts
    # 014 Oilseeds and oleaginous fruits
    # .query("~code.str.startswith('015')") # 015 Edible roots and tubers with high starch or inulin content
    .query("~code.str.startswith('016')") # 016 Stimulant, spice and aromatic crops
    # 017 Pulses (dried leguminous vegetables)
    # 018 Sugar crops
    # 019 Forage products; fibre crops; plants used in perfumery, pharmacy, or for insecticidal, fungicidal or similar purposes; beet, forage plant and flower seeds; natural rubber; living plants, cut flowers and flower buds; unmanufactured tobacco; other raw vegetable materials
    .query("~code.isin(@exclude_item_codes)")
    .reset_index(drop=True)
    .assign(PNAS_region=lambda x: x['Area'].map(dic_FAO2region))
    .merge(df_crop_classification, on="Item", how="left")
    .merge(df_harv_f, on=["Harvest_cls", "PNAS_region"], how="left")
    .merge(df_rec_r, on=["Recover_cls", "PNAS_region"], how="left")
    .assign(Moisture=lambda x: x['Item'].map(dic_moisture))
    .drop(columns=["Area Code", "Area Code (M49)", "Item Code", "Element Code", "Element", "Harvest_cls", "Recover_cls"])
)


In [ ]:
df_crop_all[[
    "code", "Item", 
    # "hf", "rr", "Moisture"
]].drop_duplicates()\
    .sort_values("code")\
    .reset_index(drop=True)\
    .to_csv("../data/params/crop_uesd_in_FAO.csv", index=False)#.query("code.str.startswith('015')") 

In [ ]:
df_cropprod_DM_g = (
    df_crop_all
    .drop(columns=["Unit"])
    .pipe(lambda df: pd.concat([
        df.drop(columns=[*_years]),
        # fresh weight -> dry weight, t -> g
        df[_years].fillna(0).mul((1 - df["Moisture"] / 100) * 1e6, axis=0)
    ], axis=1))
)

In [ ]:
df_residue_DM_g = (
    df_cropprod_DM_g.copy()
    .pipe(lambda df: pd.concat([
        df.drop(columns=_years),
        # yield -> residue
        df[_years].mul(df["hf"], axis=0)
    ], axis=1))
)


In [ ]:
df_crop_harv_C = (
    df_cropprod_DM_g.copy()
    .pipe(lambda df: pd.concat([
        df.drop(columns=_years),
        # dry matter -> C
        (df[_years] + df_residue_DM_g[_years]) * 0.5
    ], axis=1))
    .drop(columns=["Moisture", "hf", "rr", "code"])
    .groupby(["PNAS_region", "Area",], as_index=False)[_years].sum()
)

## Stock

In [ ]:
dic_livestock = {'Asses': 'Asses', 'Camels': 'Camels', 'Cattle': 'Cattle and Buffaloes', 'Chickens': 'Poultry Birds', 'Goats': 'Sheep and Goats', 'Horses': 'Horses', 'Mules and hinnies': 'Mules and hinnies', 'Sheep': 'Sheep and Goats', 'Bees': 'Other rodents', 'Buffalo': 'Cattle and Buffaloes', 'Ducks': 'Poultry Birds', 'Geese': 'Poultry Birds', 'Swine / pigs': 'Swine / pigs', 'Turkeys': 'Poultry Birds', 'Rabbits and hares': 'Rabbits and hares', 'Other camelids': 'Other camelids', 'Other rodents': 'Other rodents', 'Other birds': 'Poultry Birds'}

In [ ]:
df_livestock_factor = pd.read_excel(Path("../data/params/grazing_coefficients.xlsx"), sheet_name="livestock")
df_livestock_factor = df_livestock_factor.melt(id_vars="Unnamed: 0")\
    .rename(columns={"Unnamed: 0": "Live_cls", "variable": "PNAS_region", "value": "fi"})\
    .assign(fi=lambda _df: _df["fi"]*1000)


In [ ]:
df_stock_all = (
    pd.read_csv(cfg["paths"]["fao"] / "2024/production/Production_Crops_Livestock_E_All_Data_NOFLAG.csv", encoding_errors="ignore")
    .query("Element == 'Stocks'")
    .query("Unit != '1000 No'")
    .query("`Area Code (M49)` in @FAO_m49_lst")
    .rename(columns={"Item Code (CPC)": "code"})
    .assign(code=lambda x: x["code"].str.replace("'", ""))
    [meta_cols + year_cols]
    .dropna(subset=year_cols, how="all")
    .rename(columns=dic_year_rename)
    .query('~code.str.startswith("F")')
    
    .reset_index(drop=True)
    .assign(PNAS_region=lambda x: x['Area'].map(dic_FAO2region))
    .drop(columns=["Area Code", "Area Code (M49)", "Item Code", "Element Code", "Element", ])
    
    .assign(Live_cls=lambda x: x['Item'].map(dic_livestock))    
    .merge(df_livestock_factor, on=["Live_cls", "PNAS_region"], how="left")
    
    # unit conversion: 1000 head -> 1 head
    .assign(Unit=lambda df: df["Unit"].str.replace(" An", "").str.replace("An", "1").str.replace("No", "1").astype(int))
    .pipe(lambda df: pd.concat([
        df.drop(columns=_years),
        df[_years].mul(df["Unit"], axis=0)
    ], axis=1))
    .drop(columns=["Unit"])
)


In [ ]:
df_stock_C = (
    df_stock_all
    .pipe(lambda df: pd.concat([
        df.drop(columns=_years),
        # yield -> dry-matter intake
        df[_years].mul(df["fi"]*365, axis=0)
        # dry-matter intake -> C
        .mul(0.5, axis=0)
    ], axis=1))
    .drop(columns=["Live_cls", "fi", "code"])
    
    .groupby(["PNAS_region", "Area",], as_index=False)[_years].sum()
    .sort_values("Area")
)

## Slaughter

In [ ]:
dic_liveslaughter = {'Edible offal of cattle, fresh, chilled or frozen': 'Cattle and Buffaloes', 'Edible offal of goat, fresh, chilled or frozen': 'Sheep and Goats', 'Edible offal of sheep, fresh, chilled or frozen': 'Sheep and Goats', 'Edible offals of camels and other camelids, fresh, chilled or frozen': 'Camels', 'Meat of camels, fresh or chilled': 'Camels', 'Meat of cattle with the bone, fresh or chilled': 'Cattle and Buffaloes', 'Meat of chickens, fresh or chilled': 'Poultry Birds', 'Meat of goat, fresh or chilled': 'Sheep and Goats', 'Meat of sheep, fresh or chilled': 'Sheep and Goats', 'Raw hides and skins of cattle': 'Cattle and Buffaloes', 'Raw hides and skins of goats or kids': 'Sheep and Goats', 'Raw hides and skins of sheep or lambs': 'Sheep and Goats', 'Edible offal of buffalo, fresh, chilled or frozen': 'Cattle and Buffaloes', 'Edible offal of pigs, fresh, chilled or frozen': 'Swine / pigs', 'Edible offals of horses and other equines,  fresh, chilled or frozen': 'Horses', 'Horse meat, fresh or chilled': 'Horses', 'Meat of asses, fresh or chilled': 'Asses', 'Meat of buffalo, fresh or chilled': 'Cattle and Buffaloes', 'Meat of pig with the bone, fresh or chilled': 'Swine / pigs', 'Meat of turkeys, fresh or chilled': 'Poultry Birds', 'Raw hides and skins of buffaloes': 'Cattle and Buffaloes', 'Meat of rabbits and hares, fresh or chilled': 'Rabbits and hares', 'Meat of ducks, fresh or chilled': 'Poultry Birds', 'Meat of geese, fresh or chilled': 'Poultry Birds', 'Meat of mules, fresh or chilled': 'Mules and hinnies', 'Game meat, fresh, chilled or frozen': 'Rabbits and hares', 'Meat of other domestic camelids, fresh or chilled': 'Other camelids', 'Meat of other domestic rodents, fresh or chilled': 'Other rodents', 'Meat of pigeons and other birds n.e.c., fresh, chilled or frozen': 'Poultry Birds', 'Other meat of mammals, fresh or chilled': 'Rabbits and hares'}
# Species-specific daily feed intake (kg DM/head/day) --> g DM/head/day
df_livestock_factor = pd.read_excel(Path("../data/params/grazing_coefficients.xlsx"), sheet_name="livestock")
df_livestock_factor = df_livestock_factor.melt(id_vars="Unnamed: 0")\
    .rename(columns={"Unnamed: 0": "Live_cls", "variable": "PNAS_region", "value": "fi"})\
    .assign(fi=lambda _df: _df["fi"]*1000)

In [ ]:
df_slaughter_all = (
    pd.read_csv(cfg["paths"]["fao"] / "2024/production/Production_Crops_Livestock_E_All_Data_NOFLAG.csv", encoding_errors="ignore")
    .query("Element == 'Producing Animals/Slaughtered'")
    .query("Unit != '1000 No'")
    .query("`Area Code (M49)` in @FAO_m49_lst")
    .rename(columns={"Item Code (CPC)": "code"})
    .assign(code=lambda x: x["code"].str.replace("'", ""))
    [meta_cols + year_cols]
    .dropna(subset=year_cols, how="all")
    .rename(columns=dic_year_rename)
    .query('~code.str.startswith("215")')\
    .query('~code.str.startswith("F")')
    
    .reset_index(drop=True)
    .assign(PNAS_region=lambda x: x['Area'].map(dic_FAO2region))
    .drop(columns=["Area Code", "Area Code (M49)", "Item Code", "Element Code", "Element", ])
    
    .assign(Live_cls=lambda x: x['Item'].map(dic_liveslaughter))
    .merge(df_livestock_factor, on=["Live_cls", "PNAS_region"], how="left")
    
    # unit conversion: 1000 head -> 1 head
    .assign(Unit=lambda df: df["Unit"].str.replace(" An", "").str.replace("An", "1").str.replace("No", "1").astype(int))
    .pipe(lambda df: pd.concat([
        df.drop(columns=_years),
        df[_years].mul(df["Unit"], axis=0)
    ], axis=1))
    .drop(columns=["Unit"])
)

In [ ]:
df_slaughter_C = (
    df_slaughter_all
    .pipe(lambda df: pd.concat([
        df.drop(columns=_years),
        # yield -> dry-matter intake (fed half a year)
        df[_years].mul(df["fi"]*365/2, axis=0)
        # dry-matter intake -> C
        .mul(0.5, axis=0)
    ], axis=1))
    .drop(columns=["Live_cls", "fi", "code"])
    
    .groupby(["PNAS_region", "Area",], as_index=False)[_years].sum()
    .sort_values("Area")
)

## Merge livestock and slaughter

In [ ]:
df_stock_slaughter = df_stock_C.set_index(["PNAS_region", "Area"]).add(df_slaughter_C.set_index(["PNAS_region", "Area"]), fill_value=0)


## Food Balance Sheet

In [ ]:
df_FBS = pd.read_csv(cfg["paths"]["fao"] / "FoodBalanceSheets_E_All_Data/FoodBalanceSheets_E_All_Data_NOFLAG.csv")\
    .query("Element == 'Feed'")


In [ ]:
_moisture = {
    "Cereals":           0.14,
    "Roots":             0.75,
    "Sugar":             0.75, 
    "Pulses":            0.12,
    "Oilcrops":          0.09,
    "Vegetables":        0.90,
    "Fruits":            0.85,
    "Other Agriculture": 0.10,  # coffee/cocoa/tea, mostly dry products
}

In [ ]:
_years_FBS = list(range(2010, 2024))

In [ ]:
df_FBS_cls = pd.read_excel(Path("../data/params/FBS_feed_items.xlsx"), sheet_name="FBS_cls")

df_FBS = (
    pd.read_csv(cfg["paths"]["fao"] / "FoodBalanceSheets_E_All_Data/FoodBalanceSheets_E_All_Data_NOFLAG.csv")
        .assign(PNAS_region=lambda x: x['Area'].map(dic_FAO2region))
        .query("`Area Code (M49)` in @FAO_m49_lst")
        .rename(columns=dic_year_rename)
        .query("Element == 'Feed'")
        .merge(df_FBS_cls.drop(columns=["Item Code"]), on="Item", how="left")
        .query("Feed_cls != 'Exclude'")
        .assign(Moisture=lambda x: x['Feed_cls'].map(_moisture))
        # Unit is always 1000 t
        .pipe(lambda df: pd.concat([
            df.drop(columns=_years_FBS),
            df[_years_FBS].fillna(0)
                .mul(1000 * 1e6) # 1000 t -> g
                .mul(1 - df["Moisture"], axis=0) # fresh weight -> dry weight
                .mul(0.5, axis=0)  # dry matter -> C
        ], axis=1))
        .pipe(lambda df: pd.concat([
            df,
            df[[2021, 2022, 2023]].mean(axis=1).rename(2024)
        ], axis=1))
)


In [ ]:
df_feed_C = (df_FBS
    .drop(columns=["Area Code", "Area Code (M49)", "Element Code", "Element", "Unit", "Feed_cls", "Moisture"])
    .groupby(["PNAS_region", "Area",], as_index=False)[_years].sum()
    .sort_values("Area")
)


## Trade 

In [ ]:
forage_lst = [
    'Clover, for forage',
    'Forage and silage, alfalfa',
    'Forage and silage, maize',
    # 'Gluten feed and meal',
    # 'Vegetable products for feed n.e.c.',
    # 'Compound feed n.e.c.',

    'Hay for forage, from legumes',
    'Hay for forage, from other crops n.e.c.',
    'Hay, non-leguminous',

    'Other forage products, n.e.c.',
    'Other grasses, for forage',
    'Other legumes, for forage',

    'Turnips, for forage',

    'Beets for fodder',
    # 'Fodder and Feeding Stuff',
    'Swedes for fodder',
    'Vegetables and roots fodder',
    
    # exclude crop by-products
    # 'Bran of barley',
    # 'Bran of buckwheat',
    # 'Bran of cereals n.e.c.',
    # 'Bran of fonio',
    # 'Bran of maize',
    # 'Bran of millet',
    # 'Bran of mixed grain',
    # 'Bran of oats',
    # 'Bran of pulses',
    # 'Bran of rice',
    # 'Bran of rye',
    # 'Bran of sorghum',
    # 'Bran of wheat',

    # 'Feed minerals',
    # 'Feed supplements',
    # 'Feed, compound, cattle',
    # 'Feed, compound, pigs',
    # 'Feed, compound, poultry',
    # 'Feed, food waste preparations',
    # 'Feed, other concentrates nes',
]

In [ ]:
df_trade_all = (
    pd.read_csv(cfg["paths"]["fao"] / "2024/trade/Trade_CropsLivestock_E_All_Data_NOFLAG.csv", encoding_errors="ignore")
        .query("`Area Code (M49)` in @FAO_m49_lst")
        .rename(columns={"Item Code (CPC)": "code"})
        .assign(code=lambda x: x["code"].str.replace("'", ""))
        
        [meta_cols + year_cols]
        .assign(PNAS_region=lambda x: x['Area'].map(dic_FAO2region))
        
        .dropna(subset=year_cols, how="all")
        .rename(columns=dic_year_rename)
        .query('~code.str.startswith("F")')
        
        .query("code.str.startswith('0191')") # forage classes
        .query("Element.str.contains('quantity', case=False)")
)

df_forage_import = df_trade_all.query("Element == 'Import quantity'")
df_forage_export = df_trade_all.query("Element == 'Export quantity'")

In [ ]:
df_feed_net_import = df_forage_import.groupby(["PNAS_region", "Area"])[_years].sum().sub(
    df_forage_export.groupby(["PNAS_region", "Area"])[_years].sum(), fill_value=0)

df_feed_net_import_C = (
    df_feed_net_import
        # t -> g
        .mul(1e6)
        # fresh weight -> dry weight
        .mul(1-0.12)
        # dry matter -> C
        .mul(0.5)
        .reset_index()
        .sort_values("Area")
        .reset_index(drop=True)
)


## Residue reused as feed (Abandoned)

## Grass

In [ ]:
df_grass_harv_C_raw = (
    df_stock_slaughter
        .subtract(df_feed_C.set_index(["PNAS_region", "Area"]), fill_value=0)
        .subtract(df_feed_net_import_C.set_index(["PNAS_region", "Area"]), fill_value=0)
        .clip(lower=0)
        .reset_index()
        .sort_values("Area")
        .reset_index(drop=True)
)


### Compute the national grassland NPP cap

#### Download 500 m MODIS LUC and NPP (deprecated)

#### Compute the theoretical cap

### Correct grassland harvest

In [ ]:
df_npp_grass = pd.read_csv(cfg["paths"]["output"] / "grass_npp_2010_2024.csv", index_col=0)\
    .rename(columns=lambda x: int(x) if x.isdigit() else x)\
    .rename(columns={"FAO_name": "Area"})
df_grass_area = pd.read_csv(cfg["paths"]["output"] / "grass_area_2010_2024.csv", index_col=0)\
    .rename(columns=lambda x: int(x) if x.isdigit() else x)\
    .rename(columns={"FAO_name": "Area"})

In [ ]:
df_grass_harv_C_for_revise = df_grass_harv_C_raw.set_index(["PNAS_region", "Area"])[_years]
df_npp_grass_for_revised = df_npp_grass.set_index(["PNAS_region", "Area"])[_years].loc[df_grass_harv_C_for_revise.index.values]
df_grass_harv_C = df_grass_harv_C_for_revise.clip(upper=df_npp_grass_for_revised).reset_index()

## Wood Foresty

In [ ]:
wood_lst = ["Roundwood, coniferous", "Roundwood, non-coniferous"]

In [ ]:
df_forest_recover_rate = (pd.read_csv(Path("../data/params/forest_factors.csv"))
    [["PNAS_region", "rec_coniferous", "rec_broadleaf"]]
    .rename(columns={"rec_coniferous": "coniferous", "rec_broadleaf": "non-coniferous"})
    .melt(id_vars=["PNAS_region"]).rename(columns={"variable": "Item", "value": "rec_r"}))

df_forest_wood_rate = (pd.read_csv(Path("../data/params/forest_factors.csv"))
    [["PNAS_region", "wood_coniferous", "wood_broadleaf"]]
    .rename(columns={"wood_coniferous": "coniferous", "wood_broadleaf": "non-coniferous"})
    .melt(id_vars=["PNAS_region"]).rename(columns={"variable": "Item", "value": "wood_r"}))


In [ ]:
df_wood_all = (
    pd.read_csv(cfg["paths"]["fao"] / "2024/Forestry/Forestry_E_All_Data_NOFLAG.csv", encoding_errors="ignore")
    .query("Element == 'Production'")
    .query("Item in @wood_lst")
    .query("`Area Code (M49)` in @FAO_m49_lst")
    [["Area", "Item"] + year_cols]
    .rename(columns=dic_year_rename)
    
    .reset_index(drop=True)
    .assign(PNAS_region=lambda x: x['Area'].map(dic_FAO2region))
    .assign(Item=lambda x: x["Item"].str.replace("Roundwood, ", ""))
)



In [ ]:
df_forestry_harv_C = (
    df_wood_all
        .merge(df_forest_recover_rate, on=["PNAS_region", "Item"], how="left")
        .merge(df_forest_wood_rate, on=["PNAS_region", "Item"], how="left")
        .pipe(lambda df: pd.concat([
            df.drop(columns=_years),
            df[_years]
                .fillna(0)
                # FAO reports Removals (roundwood taken out); under HANPP, forest HANPPharv should be Fellings (total felled biomass, incl. residues left on site), since those trees are killed and their NPP terminated
                .divide(df["rec_r"], axis=0)
                # harvestable fraction -> wood fraction
                .mul(df["wood_r"], axis=0)
                # t -> g
                .mul(1e6)
                # C content
                .mul(0.5)
        ], axis=1))
        .drop(columns=["rec_r", "wood_r"])
        .groupby(["PNAS_region", "Area",], as_index=False)[_years].sum()
        .sort_values("Area")
        .reset_index(drop=True)
)


## HARV

In [ ]:
df_harv_C_total = df_crop_harv_C.set_index(["PNAS_region", "Area"]).fillna(0)\
    .add(df_grass_harv_C.set_index(["PNAS_region", "Area"]).fillna(0), fill_value=0)\
    .add(df_forestry_harv_C.set_index(["PNAS_region", "Area"]).fillna(0), fill_value=0)\
    .reset_index()

## HARV unit

In [ ]:
df_crop_harv_C_unit = (
    df_crop_harv_C
        .assign(spheric_area=lambda x: x['Area'].map(
            gdf_world_merged.set_index("FAO_name")["spheric_area"].to_dict()))
        .pipe(lambda df: pd.concat([
            df.drop(columns=_years),
            df[_years].div(df["spheric_area"], axis=0) # g -> g/m2
        ], axis=1))
)
df_crop_harv_C_unit.query("Area == 'China'")

In [ ]:
df_grass_harv_C_unit = (
    df_grass_harv_C
        .assign(spheric_area=lambda x: x['Area'].map(
            gdf_world_merged.set_index("FAO_name")["spheric_area"].to_dict()))
        .pipe(lambda df: pd.concat([
            df.drop(columns=_years),
            df[_years].div(df["spheric_area"], axis=0) # g -> g/m2
        ], axis=1))
)

In [ ]:
df_forestry_harv_C_unit = (
    df_forestry_harv_C
        .assign(spheric_area=lambda x: x['Area'].map(
            gdf_world_merged.set_index("FAO_name")["spheric_area"].to_dict()))
        .pipe(lambda df: pd.concat([
            df.drop(columns=_years),
            df[_years].div(df["spheric_area"], axis=0) # g -> g/m2
        ], axis=1))
)

In [ ]:
df_harvC_unit = (
    df_harv_C_total
        .assign(spheric_area=lambda x: x['Area'].map(
            gdf_world_merged.set_index("FAO_name")["spheric_area"].to_dict()))
        .pipe(lambda df: pd.concat([
            df.drop(columns=_years),
            df[_years].div(df["spheric_area"], axis=0) # g -> g/m2
        ], axis=1))
)
df_harvC_unit.query("Area == 'China'")

In [ ]:
path_total = (cfg["paths"]["output"] / "total")
path_total.mkdir(exist_ok=True)
path_unit = (cfg["paths"]["output"] / "unit")
path_unit.mkdir(exist_ok=True)

In [ ]:
df_crop_harv_C.to_csv(path_total / "crop.csv")
df_forestry_harv_C.to_csv(path_total / "forest.csv")
df_grass_harv_C.to_csv(path_total / "graze.csv")

df_crop_harv_C_unit.to_csv(path_unit / "crop.csv")
df_forestry_harv_C_unit.to_csv(path_unit / "forest.csv")    
df_grass_harv_C_unit.to_csv(path_unit / "graze.csv")